In [9]:
import os, json
from tqdm import tqdm
import re
from thefuzz import fuzz
from pprint import pprint
import random

In [2]:
class SectionSplitter:

    @classmethod
    def get_split_ids_mads(cls, segments):
        split_ids = []

        intros = [
            "Mads og monopolet på P4".lower(),
            "Du lytter til Mads og Monopolet".lower(),
        ]

        for i, section_dict in enumerate(segments):
            segment_text = section_dict['text'].lower().strip()


            if any([segment_text.startswith(search_intro) for search_intro in intros]):
                split_ids.append([section_dict['id']])

            elif segment_text.startswith("Mads og Monopolet".lower()):
                if i < len(segments) - 2:
                    full_string = section_dict["text"] + " " + segments[i + 1]["text"]
                    full_string = re.sub(r"\s+", " ", full_string.strip())
                    full_string = full_string.lower()

                    if full_string.startswith("Mads og Monopolet på".lower()):
                        if [section_dict['id'], segments[i + 1]['id']] not in split_ids:
                            split_ids.append([section_dict['id'], segments[i + 1]['id']])
        return split_ids


    @classmethod
    def get_split_ids(cls, segments, host):
        segments = sorted(segments, key=lambda x: x['start'])

        if host == "sara":
            return cls.get_split_ids_sara(segments)

        if host == "mads":
            return cls.get_split_ids_mads(segments)

        print("Available Hosts are: sara, mads")

    @classmethod
    def get_split_ids_sara(cls, segments):
        split_ids = []

        for i, section_dict in enumerate(segments):

            segment_text = section_dict["text"].strip().lower()

            intros = [
                "Du lytter til Sara og Monopolet podcast".lower(),
                "Tak, Du lytter til Sara og Monopolet podcast".lower(),
                "Du lytter til Sara og partilederne".lower(),
                "du lytter til sara og partilederne",
                "Du lytter til Sara og Monopolet".lower(),
                "Tak, Du lytter til Sara og Monopolet".lower(),
                "Du lytter til Sara og Monopole".lower(),
                "Tak, Du lytter til Sara og Monopole".lower(),

            ]


            if any([segment_text.startswith(intro_text) for intro_text in intros]):
                split_ids.append([section_dict['id']])

            elif segment_text.startswith("du lytter") or segment_text.startswith("tak, du lytter"):
                if i < len(segments) - 2:
                    full_string = section_dict["text"] + " " + segments[i + 1]["text"]
                    full_string = re.sub(r"\s+", " ", full_string.strip())
                    full_string = full_string.lower()
                    if any([full_string.startswith(into_text) for into_text in intros]):

                        split_ids.append([section_dict['id'], segments[i + 1]['id']])


        if len(split_ids) <= 10:
            for i, section_dict in enumerate(segments):

                if i != len(segments) - 1:

                    if any([
                        fuzz.ratio(section_dict["text"].strip().lower(), intro_text) >= 90 for intro_text in intros
                    ]):

                  #  if fuzz.ratio(section_dict["text"].strip().lower(), "Du lytter til Sara og Monopolet podcast".lower()) >= 90:
                        if [section_dict['id']] not in split_ids and "du har" not in section_dict["text"].strip().lower():
                            split_ids.append([section_dict['id']])

                  #  elif fuzz.ratio(section_dict["text"].strip().lower(), "Tak, du lytter til Sara og Monopolet podcast".lower()) >= 90:
                  #      if [section_dict['id']] not in split_ids and "du har" not in section_dict["text"].strip().lower():
                  #          split_ids.append([section_dict['id']])

                    elif section_dict["text"].strip().lower().startswith("du lytter") or section_dict["text"].strip().lower().startswith("tak, du lytter"):
                        if i < len(segments) - 2:
                            full_string = section_dict["text"] + " " + segments[i + 1]["text"]
                            full_string = re.sub(r"\s+", " ", full_string.strip())
                            full_string = full_string.lower()

                            if any([
                                fuzz.ratio(full_string, intro_text) >= 90 for intro_text in intros
                            ]):
                                if [section_dict['id'], segments[i + 1]['id']] not in split_ids and "du har" not in section_dict["text"].strip().lower():
                                    split_ids.append([section_dict['id'], segments[i + 1]['id']])

            split_ids = list(sorted(split_ids, key=lambda x: x[0]))

        return split_ids

In [3]:
# separate data per host
n_sections_dists = {"sara": {}, "mads": {}}


LOADDIR = "Transcript -- hviske-v3-conversation -- sara-og-monopolet"

# use different rules per host, as the podcast structure is diffirent
for filename in tqdm(os.listdir(LOADDIR)):
    if filename.endswith(".json"):
        data = json.load(open(os.path.join(LOADDIR, filename), "r"))
        host = "mads" if "mads" in filename else "sara"
        sections_ids = SectionSplitter.get_split_ids(segments=data["segments"], host=host)

        data["split_sections_ids"] = sections_ids
        n_sections_dists[host][filename] = len(sections_ids)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2705/2705 [00:08<00:00, 310.26it/s]


In [61]:
# Parse descriptions: extract hosts, monopolet, dilemmas

def split_by_separators(text, separators):

    escaped_seps = [re.escape(sep) for sep in separators]

    pattern = '|'.join(escaped_seps)

    parts = re.split(pattern, text)

    return [p.strip() for p in parts if p.strip()]

def parse_descriptions_sara(description):

    host = None
    
    if "Vært:" in description:
        host = description[description.index("Vært:") + len("Vært:"):]
        host = host.split("\n")[0].strip()

    raw_monopolet = None
    parsed_monopolet = None
    if "Monopolet:" in description:
        raw_monopolet = description[description.index("Monopolet:")+ len("Monopolet:"):]
        raw_monopolet = raw_monopolet.split("\n")[0].strip()

        parsed_monopolet = split_by_separators(raw_monopolet, separators=[",", " og"])

    dilemmas = []
    if "Dilemmaliste:" in description:
        dilemmas = description.split("Dilemmaliste:")[1].strip()
        dilemmas = re.findall(r'\d+\.\s*(.*?)(?=\n\d+\.|$)', description, re.DOTALL)
        dilemmas = [d.strip() for d in dilemmas]

        if "Monopolet:" in dilemmas[-1]:
            dilemmas[-1] = dilemmas[-1][:dilemmas[-1].index("Monopolet:")]

        if "Vært:" in dilemmas[-1]:
            dilemmas[-1] = dilemmas[-1][:dilemmas[-1].index("Vært:")]
            

    dilemmas = [re.sub(r'^\s*\d+[.)]\s*', '', x).strip() for x in dilemmas]
    return {
        "raw_monopolet": raw_monopolet, 
        "parsed_monopolet": parsed_monopolet,
        "host": host,
        "dilemmas": dilemmas
    }  


def parse_descriptions_mads(description):

    host = None
    
    if "Vært:" in description:
        host = description[description.index("Vært:") + len("Vært:"):]
        host = host.split("\n")[0].strip()

    raw_monopolet = None
    parsed_monopolet = None

    dilemmas_sep = ["Dilemmaer", "Dilemmaliste", "Beskrivelse", "Ugens dilemmaer"]


    for monoplet_sep in ["Monopolet:", "Panel:"]:
        
        if monoplet_sep in description:
            raw_monopolet = description[description.index(monoplet_sep) + len(monoplet_sep):]
            if "Vært:" in raw_monopolet:
                raw_monopolet = raw_monopolet[:raw_monopolet.index("Vært:")]
            for dillema_sep in dilemmas_sep + ["\n1."]:
                if dillema_sep in raw_monopolet:
                    raw_monopolet = raw_monopolet[:raw_monopolet.index(dillema_sep)]
            
            raw_monopolet = raw_monopolet.strip()
            parsed_monopolet = split_by_separators(raw_monopolet, separators=[",", " og"])
            break


    else:
        if "Vært:" in description and not any([
            x in description for x in dilemmas_sep + ["\n1."]
        ]):
            raw_monopolet = description
            if "Vært:" in raw_monopolet:
                raw_monopolet = raw_monopolet[:raw_monopolet.index("Vært:")]
            raw_monopolet = raw_monopolet.strip()
            parsed_monopolet = split_by_separators(raw_monopolet, separators=[",", " og"])


    

        
    dilemmas = []

    if any(
        [
            x + ":" in description for x in dilemmas_sep
        ]
    ):
        for dilemma_sep in dilemmas_sep:
            
            if dilemma_sep + ":" in description:
                dilemmas = description[description.index(dilemma_sep + ":") + len(dilemma_sep + ":"):].strip()
                if "\n1." in dilemmas:
                    dilemmas = re.findall(r'\d+\.\s*(.*?)(?=\n\d+\.|$)', description, re.DOTALL)
                elif "\n1)" in dilemmas:
                    dilemmas = re.findall(r'\d+\)\s*(.*?)(?=\n\d+\.|$)', description, re.DOTALL)
                else:
                    dilemmas = dilemmas.split("\n")
                    
                dilemmas = [d.strip() for d in dilemmas]

                if not dilemmas:
                    print(description)
                else:
                    if "Monopolet:" in dilemmas[-1]:
                        dilemmas[-1] = dilemmas[-1][:dilemmas[-1].index("Monopolet:")]
                    if "Panel:" in dilemmas[-1]:
                        dilemmas[-1] = dilemmas[-1][:dilemmas[-1].index("Panel:")]
            
                    if "Vært:" in dilemmas[-1]:
                        dilemmas[-1] = dilemmas[-1][:dilemmas[-1].index("Vært:")]
                break
                
    else:

        if "\n1." in description:
        
            dilemmas = re.findall(r'\d+\.\s*(.*?)(?=\n\d+\.|$)', description, re.DOTALL)
            dilemmas = [d.strip() for d in dilemmas]
    
            if "Monopolet:" in dilemmas[-1]:
                dilemmas[-1] = dilemmas[-1][:dilemmas[-1].index("Monopolet:")]
    
            if "Vært:" in dilemmas[-1]:
                dilemmas[-1] = dilemmas[-1][:dilemmas[-1].index("Vært:")]
    
            if "Monopolet:" in dilemmas[0]:
                dilemmas[-1] = dilemmas[0][:dilemmas[0].index("Monopolet:")]
    
            if "Vært:" in dilemmas[0]:
                dilemmas[-1] = dilemmas[0][:dilemmas[0].index("Vært:")]

    dilemmas = [re.sub(r'^\s*\d+[.)]\s*', '', x).strip() for x in dilemmas]        

    return {
        "raw_monopolet": raw_monopolet, 
        "parsed_monopolet": parsed_monopolet,
        "host": host,
        "dilemmas": dilemmas
    }  
    

In [5]:
# load metadata and parse

raw_descriptions = {}
for filename in os.listdir("episodes_data/"):
    if filename.endswith(".json"):
        raw_descriptions[filename] = json.load(open(os.path.join("episodes_data", filename), "r"))["description"]

In [55]:
random_sara_file = random.choice(list(n_sections_dists["sara"].keys()))
print(random_sara_file)

pprint(raw_descriptions[random_sara_file])
print()
pprint(parse_descriptions_mads(raw_descriptions[random_sara_file]))

2024-12-07-loerdag-7-dec-2024-15452448050.json
('Monopolet: Kok Dak Gjørtler Wichangoen,  radio - og tv-vært Simon Jul og '
 'skuespiller Peter Mygind.\n'
 'Vært: Sara Bro.\n'
 'Dilemmaliste: \n'
 '1. Louise og hendes kæreste, Anton, vil gerne holde jul sammen i år, men de '
 'er uenige om, hvorvidt deres lejlighed er stor nok til at kunne rumme deres '
 'to familier.\n'
 '2. Frederik oplever, at hans forældre er unødvendigt kritiske overfor hans '
 'hustru, Amalies arbejde i deres familievirksomhed, som de alle arbejder '
 'sammen i.\n'
 '3. Victoria er skruk og vil gerne have børn nu, men hun er ikke færdig med '
 'sin uddannelse, og frygter, at det vil få konsekvenser for hendes karriere '
 'efterfølgende.\n'
 '4. Sandra er bekymret for sin danselærers helbred, og er i tvivl om hun kan '
 'bede ham om, at lukke danseskolen.\n'
 '5. Laura har altid haft et velfungerende og luftigt tarmsystem, men har '
 'svært ved, hvordan hun skal tackle det, når hun er sammen med sin nye '
 'kærest

In [62]:
desriptions = {}
for filename in os.listdir("episodes_data/"):
    if filename.endswith(".json"):
        if "sara" in filename:
            desriptions[filename] = parse_descriptions_sara(
                json.load(open(os.path.join("episodes_data", filename), "r"))["description"]
            )
        else:
            desriptions[filename] = parse_descriptions_mads(
                json.load(open(os.path.join("episodes_data", filename), "r"))["description"]
            )

In [71]:
random_sara_file = random.choice(list(n_sections_dists["mads"].keys()))
print(random_sara_file)

print(raw_descriptions[random_sara_file])
print()
desriptions[random_sara_file]

2019-11-09-mads-og-monopolet-14981912645.json
Monopolet: Johanne Schmidt-Nielsen, Ane Cortzen og Eske Willerslev. 
Vært: Mads Steffensen.
Dilemmaliste: 
1. Stine overvejer at blive besøgsven for en ældre mand.
2. Mette har en ven i Tanzania, som nu flere gange bedt Mette overføre penge til ham. 
3. Katrine vil kalde sin hund for Else - men hendes nabo hedder også Else. 
4. Sisse fortalte som barn sin skoleleder, at hendes mor havde et alkoholmisbrug, men skolelederen reagerede aldrig. 
5. Kristina har mødt sin drømmemand, men han stinker af sved. 
6. Anders oplever, at cyklister bliver irriterede, hvis han ligger på hjul på cykelstien. 
7. Sofies kæreste gamer 2-3 timer hver dag, og det frustrerer hende. 
8. Maj og Torben er stærkt generet af naboens brændeovns-røg. 
9. Kalle og hans studiegruppe lånte en medstuderendes lejlighed til et filmprojekt, og i den forbindelse brændte en stjernekaster hul i en bordplade. 
10. Sofie har mødt en irsk fyr i Australien, men nu er hun kommet til a

{'raw_monopolet': 'Johanne Schmidt-Nielsen, Ane Cortzen og Eske Willerslev.',
 'parsed_monopolet': ['Johanne Schmidt-Nielsen',
  'Ane Cortzen',
  'Eske Willerslev.'],
 'host': 'Mads Steffensen.',
 'dilemmas': ['Stine overvejer at blive besøgsven for en ældre mand.',
  'Mette har en ven i Tanzania, som nu flere gange bedt Mette overføre penge til ham.',
  'Katrine vil kalde sin hund for Else - men hendes nabo hedder også Else.',
  'Sisse fortalte som barn sin skoleleder, at hendes mor havde et alkoholmisbrug, men skolelederen reagerede aldrig.',
  'Kristina har mødt sin drømmemand, men han stinker af sved.',
  'Anders oplever, at cyklister bliver irriterede, hvis han ligger på hjul på cykelstien.',
  'Sofies kæreste gamer 2-3 timer hver dag, og det frustrerer hende.',
  'Maj og Torben er stærkt generet af naboens brændeovns-røg.',
  'Kalle og hans studiegruppe lånte en medstuderendes lejlighed til et filmprojekt, og i den forbindelse brændte en stjernekaster hul i en bordplade.',
  'Sof

In [73]:
for k, d in desriptions.items():
    if len(d["dilemmas"]) < 10:
        print(d)
        print()

{'raw_monopolet': 'Sendt d. 13. marts 2004.\nPanelet var daværende vært på TV Avisen Paula Larrain, livsstilsekspert Mads Christensen og politiker Naser Khader, daværende medlem af Det Radikale Venstre.', 'parsed_monopolet': ['Sendt d. 13. marts 2004.\nPanelet var daværende vært på TV Avisen Paula Larrain', 'livsstilsekspert Mads Christensen', 'politiker Naser Khader', 'daværende medlem af Det Radikale Venstre.'], 'host': 'Mads Steffensen.', 'dilemmas': []}

{'raw_monopolet': 'Sendt d. 18. april 2009. I monopolet var sangerinde Annette Heick, filminstruktør & tegner Anders Morgenthaler og modeekspert Uffe Buchard.', 'parsed_monopolet': ['Sendt d. 18. april 2009. I monopolet var sangerinde Annette Heick', 'filminstruktør & tegner Anders Morgenthaler', 'modeekspert Uffe Buchard.'], 'host': 'Mads Steffensen.', 'dilemmas': []}

{'raw_monopolet': 'Sendt d. 18 september 2004. I monopolet var forfatter Suzanne Bjerrehuus, politiker Søren Pind og filminstruktør Søren Fauli.', 'parsed_monopolet

In [74]:
# check on amount of sections anomalies for Sara
# Margin is an acceptable difference between number of breaks and number of dilemmas

margin = 3

issues = {}
for filename in n_sections_dists["sara"]:
    if abs(len(desriptions[filename]["dilemmas"]) - n_sections_dists["sara"][filename]) > margin:
        issues[filename] = {"num_dilemmas": len(desriptions[filename]["dilemmas"]), "num_sections": n_sections_dists["sara"][filename]}

In [75]:
len(issues)

12

In [76]:
issues

{'2023-12-24-jul-med-sara-og-monopolet-15452348051.json': {'d': 0, 's': 10},
 '2022-12-17-foelger-op-11032200045.json': {'d': 0, 's': 7},
 '2024-07-27-sommeropfoelgninger-med-sara-og-monopolet-uge-30-15452448031.json': {'d': 0,
  's': 4},
 '2024-06-15-loerdag-15-jun-2024-15452448025.json': {'d': 0, 's': 11},
 '2021-05-29-highlights-11032112521.json': {'d': 0, 's': 9},
 '2024-12-21-foelger-op-15452448052.json': {'d': 0, 's': 8},
 '2022-10-15-sara-og-partilederne-11032200036.json': {'d': 0, 's': 8},
 '2023-06-17-loerdag-17-jun-2023-15452348024.json': {'d': 0, 's': 11},
 '2021-12-18-foelger-op-11032112550.json': {'d': 0, 's': 4},
 '2023-12-16-foelger-op-15452348050.json': {'d': 0, 's': 13},
 '2021-01-09-loerdag-9-jan-2021-11032112501.json': {'d': 9, 's': 2},
 '2023-09-09-20-aar-med-monopolet-15452348036.json': {'d': 0, 's': 18}}

In [ ]:
# Here I was manually looking into the files to see if I missed any breaks or is there is an issue with transcriptions

In [77]:
[k for k in n_sections_dists["sara"] if n_sections_dists["sara"][k] == 2]

['2024-07-20-sommeropfoelgninger-med-sara-og-monopolet-uge-29-15452448030.json',
 '2024-08-10-sommeropfoelgninger-med-sara-og-monopolet-uge-32-15452448033.json',
 '2021-01-09-loerdag-9-jan-2021-11032112501.json']

In [78]:
filename = "2024-07-20-sommeropfoelgninger-med-sara-og-monopolet-uge-29-15452448030.json"
host = "sara"
data = json.load(open(os.path.join(LOADDIR, filename), "r"))
out = SectionSplitter.get_split_ids(segments=data["segments"], host=host)

In [79]:
out

[[0], [813]]

In [ ]:
# check on amount of sections anomalies for Sara
# Margin is an acceptable difference between number of breaks and number of dilemmas

In [80]:
random_mads_file = random.choice(list(n_sections_dists["mads"].keys()))
print(random_mads_file)
pprint(desriptions[random_mads_file]["dilemmas"])
print(len(desriptions[random_mads_file]["dilemmas"]))

2020-01-11-mads-og-monopolet-14982012602.json
['Sofie skal på ferie med sin kæreste, nu har moren meddelt, at de tager på '
 'samme rejse.',
 'Lone og hendes mand er i tvivl, om de godt kan give alkoholfri øl til deres '
 '11-årige søn.',
 'Vicki kom til at kysse med sin læge og  hun har en gynækologisk undersøgelse '
 'hos ham om et par uger.',
 'Berit kan slet ikke holde sine svigerforældre ud.',
 'Preben og Berit havde morgensex og fik en seddel fra en nabo, som skrev, det '
 'var frækt at lytte med.',
 'Mathias og hans venner er uenige, om man kan levere en vin tilbage på en '
 'restaurant.',
 'Emilie er i tvivl, om hun skal have sin søsters bonusbørn med på et '
 'familiefoto.',
 'Cornelia har ikke lyst til at være sammen med sin veninde, fordi hun synes, '
 'venindens barn er uudholdeligt.',
 'Tilde har inviteret sin veninde til koncert samme dato som en veninde fra '
 'udlandet kommer på besøg.',
 'Anna arbejder i udlandet og flirter med en anden dansker, Preben, men de er '
 'b

In [81]:
desriptions["2019-06-15-mads-og-monopolet-14981912624.json"]

{'raw_monopolet': 'Mads & Monopolet på Folkemødet 2019.',
 'parsed_monopolet': ['Mads & Monopolet på Folkemødet 2019.'],
 'host': 'Mads Steffensen.',
 'dilemmas': []}

In [82]:
margin = 3

mads_issues = {}
for filename in n_sections_dists["mads"]:
    if abs(len(desriptions[filename]["dilemmas"]) - n_sections_dists["mads"][filename]) > margin:
        mads_issues[filename] = {"num_dilemmas": len(desriptions[filename]["dilemmas"]), "num_sections": n_sections_dists["mads"][filename]}

In [83]:
len(mads_issues)

51

In [84]:
mads_issues = {k: v for k,v in mads_issues.items() if v["d"] > 1}

In [85]:
len(mads_issues)

13

In [86]:
mads_issues

{'2020-12-12-mads-og-monopolet-14982012650.json': {'d': 9, 's': 13},
 '2017-12-02-mads-og-monopolet-uge-48-14981712648.json': {'d': 10, 's': 5},
 '2017-03-11-mads-og-monopolet-uge-10-14981712610.json': {'d': 19, 's': 10},
 '2018-08-25-mads-og-monopolet-14981812634.json': {'d': 10, 's': 6},
 '2017-04-22-mads-og-monopolet-uge-16-14981712616.json': {'d': 22, 's': 11},
 '2017-02-04-mads-og-monopolet-uge-5-14981712605.json': {'d': 19, 's': 11},
 '2017-09-16-mads-og-monopolet-uge-37-14981712637.json': {'d': 19, 's': 11},
 '2017-12-09-mads-og-monopolet-uge-49-14981712649.json': {'d': 19, 's': 10},
 '2016-11-12-mads-og-monopolet-uge-45-2016-14981612645.json': {'d': 10,
  's': 6},
 '2017-05-06-mads-og-monopolet-uge-18-14981712618.json': {'d': 19, 's': 8},
 '2016-09-24-mads-og-monopolet-uge-38-2016-14981612638.json': {'d': 10,
  's': 14},
 '2019-12-07-mads-og-monopolet-14981912649.json': {'d': 10, 's': 6},
 '2019-03-02-mads-og-monopolet-14981912609.json': {'d': 19, 's': 9}}

In [ ]:
# Once you are happy with the results, save kep files and the parsed dilemmas

In [87]:
keep_files = [x for x in desriptions if len(desriptions[x]["dilemmas"]) > 1]

In [88]:
savedir = "filtered_meta"

os.makedirs("filtered_meta", exist_ok=True)

with open("filtered_meta/filtered_filenames.json", "w") as f:
    json.dump(keep_files, f)

with open("filtered_meta/parsed_descriptions.json", "w") as f:
    json.dump(desriptions, f)
    

In [89]:
len(keep_files)

347